In [2]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")

y_train = pd.read_csv(
    "../data/y_train.csv"
).squeeze()

In [3]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [4]:
from sklearn.linear_model import Ridge, Lasso
from src.evaluate import rmse_cv

In [5]:
model_ridge = Ridge(alpha=10.0)
model_lasso = Lasso(alpha=0.001)

ridge_rmse = rmse_cv(
    model_ridge,
    X_train,
    y_train
).mean()

lasso_rmse = rmse_cv(
    model_lasso,
    X_train,
    y_train
).mean()

print(f"Ridge RMSE: {ridge_rmse:.5f}")
print(f"Lasso RMSE: {lasso_rmse:.5f}")

Ridge RMSE: 0.11273
Lasso RMSE: 0.11175


In [6]:
# 產生並儲存 Ridge 的 OOF 預測值
from sklearn.model_selection import cross_val_predict

print("正在計算 Ridge 5-Fold OOF 預測值...")

# 使用你上面已經定義好的 model_ridge (alpha=10.0)
oof_ridge = cross_val_predict(
    model_ridge, 
    X_train, 
    y_train, 
    cv=5
)

# 存檔備用，給 ensemble.ipynb 抓取
pd.Series(oof_ridge, name="SalePrice").to_csv("../data/ridge_oof_train.csv", index=False)
print("Ridge OOF 已成功儲存為 ../data/ridge_oof_train.csv")

正在計算 Ridge 5-Fold OOF 預測值...
Ridge OOF 已成功儲存為 ../data/ridge_oof_train.csv


In [7]:
# Linear Regression RMSE表格

import pandas as pd

result = pd.DataFrame({
    "Model":[
        "Ridge",
        "Lasso"
    ],
    "RMSE":[
        ridge_rmse,
        lasso_rmse
    ]
})

result

,Model,RMSE
0,Ridge,0.112725
1,Lasso,0.111754


In [8]:
result.to_csv(
    "../data/linear_regression_result.csv",
    index=False
)

In [9]:
# Lasso Submission

import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import RobustScaler

# 1. 讀取 preprocessing.ipynb 已經輸出的同一組資料
X_train_lasso = pd.read_csv("../data/X_train.csv")
y_train_lasso = pd.read_csv("../data/y_train.csv").squeeze("columns")
X_test_lasso = pd.read_csv("../data/X_test.csv")
test_ids = pd.read_csv("../data/test.csv", usecols=["Id"])["Id"]

# 2. 檢查 train 特徵與 target 筆數必須一致
print("X_train:", X_train_lasso.shape)
print("y_train:", y_train_lasso.shape)
print("X_test:", X_test_lasso.shape)

assert len(X_train_lasso) == len(y_train_lasso), "X_train 和 y_train 筆數不一致"
assert list(X_train_lasso.columns) == list(X_test_lasso.columns), "X_train 和 X_test 欄位不一致"

# 3. 特徵縮放 (改用 RobustScaler 對異常值更具抵抗力)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_lasso)
X_test_scaled = scaler.transform(X_test_lasso)

# 4. 訓練 LassoCV (內建 5-Fold CV 自動找最佳 alpha)
print("\n正在執行 LassoCV 5-Fold 交叉驗證...")
lasso = LassoCV(alphas=[0.0001, 0.0003, 0.0005, 0.001, 0.01, 0.1, 1], 
                cv=5, 
                max_iter=50000, 
                random_state=42)
lasso.fit(X_train_scaled, y_train_lasso)

best_alpha_index = list(lasso.alphas_).index(lasso.alpha_)
best_mse = lasso.mse_path_[best_alpha_index].mean()
print(f"-> 最佳 alpha 值: {lasso.alpha_}")
print(f"-> 5-Fold CV RMSE: {np.sqrt(best_mse):.5f}\n")

# 5. 預測後轉回原本 SalePrice 尺度
preds_log = lasso.predict(X_test_scaled)
preds = np.expm1(preds_log)

# 6. 建立 Kaggle submission
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": preds
})

# 確保資料夾存在並嘗試存檔 (包含防呆機制)
os.makedirs("../submissions", exist_ok=True)
try:
    submission.to_csv("../submissions/linear_submission.csv", index=False)
    print("成功！預測結果已儲存為 ../submissions/linear_submission.csv")
except PermissionError:
    print("【錯誤】權限被拒絕！請檢查是否在 Excel 或其他程式中開啟了 linear_submission.csv，將其關閉後再重新執行此 Cell。")

submission.head()

X_train: (1455, 292)
y_train: (1455,)
X_test: (1459, 292)

正在執行 LassoCV 5-Fold 交叉驗證...
-> 最佳 alpha 值: 0.0005
-> 5-Fold CV RMSE: 0.11042

成功！預測結果已儲存為 ../submissions/linear_submission.csv


,Id,SalePrice
0,1461,118154.011932
1,1462,164213.093803
2,1463,184210.397147
3,1464,203070.272208
4,1465,192929.150186
